# 08 — Evaluation nach Ichmoukhamedov et al. (2024)

Implementiert das automatisierte Evaluationsframework aus:
> Ichmoukhamedov, Hinns & Martens (2024). *How good is my story? Towards quantitative metrics for evaluating LLM-generated XAI narratives.* arXiv:2412.10220

Das Framework besteht aus drei Evaluationskategorien (Fig. 1 im Paper):

| Kategorie | Metrik | Status |
|-----------|--------|--------|
| **Faithfulness** | Rank Accuracy (RA), Sign Accuracy (SA), Value Accuracy (VA) | ✓ implementiert — ⚠️ Selection-Bias (→ 4.1) |
| **Assumptions** | Extraktion per LLM + Perplexität | Extraktion ✓, PPL ⚠️ (lokales LLM nötig) |
| **Human Similarity** | Kosinus-Ähnlichkeit zu Referenznarrativen | ✗ (keine Referenznarrative vorhanden) |

**Ablauf (Fig. 1):**
1. Narrative aus Pipelines 04/05/06 laden
2. Extraction LLM (Claude) extrahiert pro Feature: Rang, Vorzeichen, Wert, Annahme
3. Abgleich mit SHAP-Ground-Truth (Gl. 1 im Paper) → RA, SA, VA
4. Annahmen anzeigen (Perplexität: optionaler Aufruf mit lokalem LLM)

**Anpassungen an unser Szenario:**
- Regressionstask (Poisson) statt binärer Klassifikation
- Beiträge im Log-Raum → Vorzeichen: +1 = erhöht, −1 = senkt Vorhersage
- Feature-Werte teilweise normalisiert (temp/hum/windspeed) — VA mit Denormalisierungstoleranz

In [1]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR
from utils.explanations import FEATURE_SCHEMA
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY        = 'poisson_log'
MODEL           = DEFAULT_MODEL   # Extraction LLM (Paper: gpt-4o)
TOP_K           = 4               # Paper: top-4 Features nach |Beitrag|
PIPELINES       = ['04', '05', '06']
PIPELINE_LABELS = {'04': 'JSON→Text', '05': 'Vision', '06': 'Tool-Use'}
XAI_MODELS     = ['xgb', 'ebm']

OUT_DIR = RESULTS_DIR / 'eval08_ichmoukhamedov'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Extraction-Modell: {MODEL}')
print(f'Top-K Features:    {TOP_K}  (Paper: 4)')
print(f'Ausgabe:           {OUT_DIR}')

Extraction-Modell: claude-sonnet-4-6
Top-K Features:    4  (Paper: 4)
Ausgabe:           /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval08_ichmoukhamedov


## 1 · Daten laden

In [2]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'FEHLT: {f}'); continue
            d = json.loads(f.read_text())

            gt_path = EXPLANATIONS_DIR / f'local_{xai}_{LOSS_KEY}_inst{iid}.json'
            gt = json.loads(gt_path.read_text())

            records.append({
                'pipeline':          pipeline,
                'pipeline_label':    PIPELINE_LABELS[pipeline],
                'xai_model':         xai.upper(),
                'instance_id':       iid,
                'explanation':       d.get('explanation', ''),
                'gt_contributions':  gt['contributions'][:TOP_K],
                'gt_feature_values': gt['feature_values'],
                'gt_prediction':     gt['prediction'],
                'gt_y_true':         gt['y_true'],
            })

df = pd.DataFrame(records)
print(f'{len(df)} Narrative geladen '
      f'({len(PIPELINES)} Pipelines × {len(XAI_MODELS)} Modelle × {len(INSTANCE_IDS)} Instanzen)')
print(f'Top-{TOP_K} Ground-Truth-Features (Beispiel inst=42, XGB):')
for c in records[0]['gt_contributions']:
    print(f"  rank={records[0]['gt_contributions'].index(c)}  "
          f"{c['feature']:12s}  contrib={c['contribution']:+.4f}  val={c['value']}")

60 Narrative geladen (3 Pipelines × 2 Modelle × 10 Instanzen)
Top-4 Ground-Truth-Features (Beispiel inst=42, XGB):
  rank=0  hr            contrib=+0.2874  val=20.0
  rank=1  yr            contrib=-0.2241  val=0.0
  rank=2  temp          contrib=+0.1334  val=0.8
  rank=3  mnth          contrib=+0.1147  val=7.0


## 2 · Extraction LLM (Sec. III im Paper)

Das Paper schlägt vor, ein separates *Extraction LLM* zu nutzen, das pro Narrativ
extrahiert (Fig. 3):
- **Rank** (`rank`): 0-basierter Wichtigkeitsrang laut Narrativ
- **Sign** (`sign`): +1 (erhöhend) oder −1 (senkend)
- **Value** (`value`): explizit genannter Featurewert (oder `null`)
- **Assumption** (`assumption`): injiziertes Hintergrundwissen (ein Satz oder `"None"`)

In [3]:
EXTRACTION_SYSTEM = """\
Du bist ein Extraktionsmodell für XAI-Narrative eines Fahrradverleih-Modells.
Extrahiere die angeforderten Informationen ausschließlich aus dem Narrativ.
Antworte ausschließlich mit einem validen JSON-Objekt — kein Text, keine Markdown-Blöcke.\
"""

# Denormalisierung: normalisierte Featurewerte → menschlich lesbare Einheiten
# (Bike Sharing: temp/41=°C, hum/100=%, windspeed/67=km/h)
_DENORM = {
    'temp':      lambda v: v * 41,
    'hum':       lambda v: v * 100,
    'windspeed': lambda v: v * 67,
}


def build_extraction_prompt(explanation: str, xai_model: str) -> str:
    feat_descs = {f: FEATURE_SCHEMA[f]['description'] for f in FEATURE_SCHEMA}
    payload = {
        'aufgabe': (
            f'Extrahiere strukturierte Informationen aus dem folgenden deutschen Narrativ '
            f'zu einem {xai_model}-Regressionsmodell für einen Fahrradverleih. '
            f'Das Modell sagt stündliche Fahrrad-Ausleihen voraus. '
            f'Positive Beiträge erhöhen die Vorhersage, negative senken sie.'
        ),
        'narrativ': explanation,
        'alle_features': list(FEATURE_SCHEMA.keys()),
        'feature_beschreibungen': feat_descs,
        'extraktionsanweisung': (
            'Gib für jedes Feature, das im Narrativ als wichtig erwähnt wird, ein Objekt zurück:\n'
            '  rank: 0-basierter Wichtigkeitsrang laut Narrativ (0 = wichtigstes Feature)\n'
            '  sign: +1 wenn das Feature die Vorhersage erhöht, -1 wenn es sie senkt\n'
            '  value: numerischer Featurewert, falls explizit im Narrativ genannt, sonst null\n'
            '  assumption: einziger Satz mit Hintergrundwissen warum das Feature diesen Einfluss hat; '
            '"None" falls kein Hintergrundwissen hinzugefügt wurde'
        ),
        'ausgabeformat_beispiel': {
            'hr':   {'rank': 0, 'sign':  1, 'value': 13,   'assumption': 'Mittagszeit ist typischerweise nachfragereich.'},
            'temp': {'rank': 1, 'sign': -1, 'value': None,  'assumption': 'Kälte schreckt Radfahrer ab.'},
        },
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def parse_extraction(raw: str) -> dict:
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group())
    except json.JSONDecodeError:
        return {}


print('Extraction-Prompt und Parser definiert.')
sample = build_extraction_prompt(df.iloc[0]['explanation'][:200] + '…', 'XGB')
print(f'Prompt-Größe (Beispiel): {len(sample)} Zeichen')

Extraction-Prompt und Parser definiert.
Prompt-Größe (Beispiel): 1925 Zeichen


## 3 · LLM-Extraktionen ausführen

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env` — **30 API-Calls** (3 Pipelines × 2 Modelle × 5 Instanzen).
>
> Bereits berechnete Extraktionen werden aus `eval08_ichmoukhamedov/extractions.json` geladen (Cache).

In [4]:
CACHE_PATH = OUT_DIR / 'extractions.json'

if CACHE_PATH.exists():
    extractions = json.loads(CACHE_PATH.read_text())
    print(f'Extractions aus Cache geladen: {len(extractions)} Einträge.')
else:
    extractions = {}

missing = [
    row for _, row in df.iterrows()
    if f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}" not in extractions
]

if missing:
    print(f'{len(missing)} fehlende Extraktionen — starte API-Calls...')
    total_in, total_out = 0, 0

    for row in missing:
        key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
        prompt = build_extraction_prompt(row['explanation'], row['xai_model'])

        response = ask_text(
            prompt,
            system=EXTRACTION_SYSTEM,
            model=MODEL,
            max_tokens=700,
            cache_system=True,
        )
        usage   = response.get('usage', {})
        in_tok  = usage.get('input_tokens', 0)
        out_tok = usage.get('output_tokens', 0)
        total_in += in_tok; total_out += out_tok

        raw = response['content'][0]['text'].strip()
        extraction = parse_extraction(raw)

        extractions[key] = {
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'raw':            raw,
            'extraction':     extraction,
            'usage':          {'input_tokens': in_tok, 'output_tokens': out_tok},
        }
        print(f"  {key}: {len(extraction)} Features  in={in_tok}  out={out_tok}")

    CACHE_PATH.write_text(json.dumps(extractions, indent=2, ensure_ascii=False))
    print(f'\nGespeichert: {CACHE_PATH}')
    print(f'Gesamt: input={total_in}  output={total_out}')
else:
    print('Alle Extraktionen bereits im Cache.')

print(f'\nBeispiel-Extraktion (Pipeline 04, XGB, inst=42):')
sample_key = '04_xgb_inst42'
if sample_key in extractions:
    print(json.dumps(extractions[sample_key]['extraction'], indent=2, ensure_ascii=False))

60 fehlende Extraktionen — starte API-Calls...
  04_xgb_inst224: 4 Features  in=1277  out=286
  04_xgb_inst580: 3 Features  in=1263  out=232
  04_xgb_inst1041: 5 Features  in=1300  out=419
  04_xgb_inst1481: 5 Features  in=1305  out=391
  04_xgb_inst1677: 5 Features  in=1300  out=411
  04_xgb_inst2058: 4 Features  in=1281  out=321
  04_xgb_inst2510: 6 Features  in=1283  out=437
  04_xgb_inst3543: 6 Features  in=1312  out=465
  04_xgb_inst3847: 6 Features  in=1305  out=433
  04_xgb_inst4454: 5 Features  in=1337  out=401
  04_ebm_inst224: 3 Features  in=1223  out=236
  04_ebm_inst580: 5 Features  in=1257  out=375
  04_ebm_inst1041: 6 Features  in=1294  out=469
  04_ebm_inst1481: 5 Features  in=1280  out=413
  04_ebm_inst1677: 5 Features  in=1285  out=382
  04_ebm_inst2058: 6 Features  in=1295  out=443
  04_ebm_inst2510: 3 Features  in=1295  out=286
  04_ebm_inst3543: 4 Features  in=1286  out=339
  04_ebm_inst3847: 7 Features  in=1294  out=518
  04_ebm_inst4454: 5 Features  in=1272  out=4

## 4 · Faithfulness-Metriken (Gl. 1 im Paper)

$$X_A = \frac{\sum_{x_j \neq \phi} \delta_{x_j, x_j^*}}{n - \sum 1[x_j = \phi]}$$

- **RA** (Rank Accuracy): stimmt der extrahierte Rang mit dem tatsächlichen SHAP-Rang überein?
- **SA** (Sign Accuracy): stimmt das extrahierte Vorzeichen mit dem SHAP-Vorzeichen überein?
- **VA** (Value Accuracy): stimmt der extrahierte Featurewert mit dem tatsächlichen Wert überein?
  Werte, die nicht im Narrativ genannt wurden (`null`), werden nicht gewertet.

**Adaptation:** Featurewerte werden sowohl normalisiert (0–1) als auch denormalisiert (°C, %) verglichen,
da LLMs die Werte häufig in menschlich lesbarer Form zitieren.

In [5]:
def is_value_match(feat: str, extracted: float, gt: float, tol: float = 1.0) -> bool:
    """Vergleich mit Toleranz; prüft auch denormalisierte Einheiten (z.B. °C statt [0,1])."""
    if abs(extracted - gt) <= tol:
        return True
    if feat in _DENORM:
        dv = _DENORM[feat](gt)
        if abs(extracted - dv) <= tol:
            return True
    return False


def compute_faithfulness(extraction: dict, gt_contributions: list) -> dict:
    """
    Berechnet RA, SA, VA nach Gl. 1 aus Ichmoukhamedov et al. (2024).
    ϕ = null/nicht extrahiert → wird aus Nenner herausgerechnet.
    """
    gt_rank  = {c['feature']: i              for i, c in enumerate(gt_contributions)}
    gt_sign  = {c['feature']: (1 if c['contribution'] >= 0 else -1)
                for c in gt_contributions}
    gt_value = {c['feature']: c['value']     for c in gt_contributions}

    ra_hits, ra_n = 0, 0
    sa_hits, sa_n = 0, 0
    va_hits, va_n = 0, 0

    for feat, info in extraction.items():
        feat_key = feat.lower()
        if feat_key not in gt_rank:
            continue  # Feature nicht unter Top-K → überspringen (wie im Paper)

        r = info.get('rank')
        if r is not None:
            ra_n += 1
            try:
                if int(float(r)) == gt_rank[feat_key]:
                    ra_hits += 1
            except (ValueError, TypeError):
                pass

        s = info.get('sign')
        if s is not None:
            sa_n += 1
            try:
                if int(float(s)) == gt_sign[feat_key]:
                    sa_hits += 1
            except (ValueError, TypeError):
                pass

        v = info.get('value')
        if v is not None and str(v).lower() not in ('null', 'none', ''):
            try:
                v_float = float(v)
                gt_v    = float(gt_value.get(feat_key, 0))
                va_n += 1
                if is_value_match(feat_key, v_float, gt_v):
                    va_hits += 1
            except (ValueError, TypeError):
                pass

    return {
        'RA': round(ra_hits / ra_n, 4) if ra_n > 0 else None,
        'SA': round(sa_hits / sa_n, 4) if sa_n > 0 else None,
        'VA': round(va_hits / va_n, 4) if va_n > 0 else None,
        'RA_hits': ra_hits, 'RA_n': ra_n,
        'SA_hits': sa_hits, 'SA_n': sa_n,
        'VA_hits': va_hits, 'VA_n': va_n,
        'n_extracted': len(extraction),
    }


faith_rows = []
for _, row in df.iterrows():
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    ext = extractions.get(key, {}).get('extraction', {})
    m   = compute_faithfulness(ext, row['gt_contributions'])
    faith_rows.append({
        'pipeline_label': row['pipeline_label'],
        'xai_model':      row['xai_model'],
        'instance_id':    row['instance_id'],
        **m,
    })

faith_df = pd.DataFrame(faith_rows)

summary = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3)
print('Ø Faithfulness-Metriken (Gl. 1, Ichmoukhamedov et al. 2024):')
display(summary)

print()
print('Aufgeschlüsselt nach Modell:')
display(faith_df.groupby(['pipeline_label', 'xai_model'])[['RA', 'SA', 'VA']].mean().round(3))

Ø Faithfulness-Metriken (Gl. 1, Ichmoukhamedov et al. 2024):


,RA,SA,VA
pipeline_label,,,
JSON→Text,0.846,0.988,1.0
Tool-Use,0.883,1.000,1.0
Vision,0.788,0.983,1.0



Aufgeschlüsselt nach Modell:


RA     SA   VA
pipeline_label xai_model                   
JSON→Text      EBM        0.975  0.975  1.0
               XGB        0.717  1.000  1.0
Tool-Use       EBM        0.933  1.000  1.0
               XGB        0.833  1.000  1.0
Vision         EBM        0.625  0.967  1.0
               XGB        0.950  1.000  1.0

In [6]:
# Detail-Tabelle: absolute Trefferquoten
detail = faith_df.groupby('pipeline_label').agg(
    RA_hits=('RA_hits', 'sum'), RA_n=('RA_n', 'sum'),
    SA_hits=('SA_hits', 'sum'), SA_n=('SA_n', 'sum'),
    VA_hits=('VA_hits', 'sum'), VA_n=('VA_n', 'sum'),
    n_extracted=('n_extracted', 'mean'),
)
detail['RA (abs)'] = detail.apply(lambda r: f"{int(r.RA_hits)}/{int(r.RA_n)}", axis=1)
detail['SA (abs)'] = detail.apply(lambda r: f"{int(r.SA_hits)}/{int(r.SA_n)}", axis=1)
detail['VA (abs)'] = detail.apply(lambda r: f"{int(r.VA_hits)}/{int(r.VA_n)}", axis=1)
print('Absolute Treffer (Σ über alle 10 Narrative pro Pipeline):')
display(detail[['RA (abs)', 'SA (abs)', 'VA (abs)', 'n_extracted']].rename(
    columns={'n_extracted': 'Ø extrahierte Features'}
))

Absolute Treffer (Σ über alle 10 Narrative pro Pipeline):


,RA (abs),SA (abs),VA (abs),Ø extrahierte Features
pipeline_label,,,,
JSON→Text,61/72,71/72,70/70,4.9
Tool-Use,54/62,62/62,62/62,3.4
Vision,56/71,70/71,68/68,5.6


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
metric_labels = [
    'RA — Rang-Genauigkeit',
    'SA — Vorzeichen-Genauigkeit',
    'VA — Wert-Genauigkeit',
]
metric_cols = ['RA', 'SA', 'VA']
labels  = [PIPELINE_LABELS[p] for p in PIPELINES]
colors  = ['#4c72b0', '#dd8452', '#55a868']

for ax, col, title in zip(axes, metric_cols, metric_labels):
    vals = [
        faith_df[faith_df.pipeline_label == PIPELINE_LABELS[p]][col].mean()
        for p in PIPELINES
    ]
    bars = ax.bar(labels, vals, color=colors, width=0.5)
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=10, pad=8)
    ax.set_ylabel('Genauigkeit (0–1)')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.35, linewidth=1)
    for bar, val in zip(bars, vals):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(
    'Faithfulness-Metriken nach Ichmoukhamedov et al. (2024) — Gl. 1',
    y=1.03, fontsize=12
)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval08_faithfulness.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval08_faithfulness.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_95670/2288679335.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.1 · Methodische Einschränkung: Selection-Bias der Metriken

> **Wichtiger Hinweis zur Interpretation von RA, SA, VA:**

Die Formel iteriert über `extraction.items()` – also nur über Features,
die das LLM **selbst im Narrativ erwähnt hat**. Features aus den Top-K,
die das LLM *nicht* erwähnte, werden **nicht als Fehler gezählt**
(sie fehlen schlicht im Nenner).

**Konsequenz:**
- Ein LLM, das nur 1 Feature erwähnt und dabei das Vorzeichen korrekt trifft,
  erzielt SA = 1.0 – genauso wie ein LLM, das alle 4 Features korrekt beschreibt.
- Die Metriken messen **Präzision, nicht Recall** der Feature-Beschreibung.
- SA = 1.0 über alle Pipelines bedeutet: *was erwähnt wurde, hatte das richtige Vorzeichen* –
  nicht: *alle wichtigen Features wurden erwähnt*.

**Fairerer Ansatz** (hier nicht implementiert, da Paper-konform):
Iteration über `gt_contributions` (alle Top-K), mit Strafe für nicht erwähnte Features.
Dies würde insbesondere JSON→Text (direkter Zugang zu Zahlen) weniger bevorteilen.


## 5 · Assumptions (Plausibilität)

Das Extraction LLM extrahiert pro Feature eine **Annahme** (assumption): einen Satz,
der das injizierte Hintergrundwissen des Generation LLM beschreibt.

Beispiel aus dem Paper (Fig. 3):
> *"Scoring goals is a key factor in determining a team's performance and the likelihood of a player standing out."*

Das Paper bewertet Annahmen mit **Perplexität** (Gl. 2) relativ zu Llama-3-8B oder Mistral-7B.
Da kein lokales LLM verfügbar ist, werden die Annahmen hier extrahiert und angezeigt (Perplexität: Zelle 15).

In [8]:
assumption_rows = []
for _, row in df.iterrows():
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    ext = extractions.get(key, {}).get('extraction', {})
    for feat, info in ext.items():
        assumption = info.get('assumption', 'None')
        if assumption and assumption.lower() not in ('none', 'null', ''):
            assumption_rows.append({
                'pipeline_label': row['pipeline_label'],
                'xai_model':      row['xai_model'],
                'instance_id':    row['instance_id'],
                'feature':        feat.lower(),
                'assumption':     assumption,
            })

assumption_df = pd.DataFrame(assumption_rows)
assumption_df.to_csv(OUT_DIR / 'assumptions.csv', index=False)

print(f'Extrahierte Annahmen gesamt: {len(assumption_df)}')
print(f'Ø Annahmen pro Narrativ:     {len(assumption_df)/len(df):.2f}')
print(f'Gespeichert: {OUT_DIR / "assumptions.csv"}')
print()

# Annahmen pro Pipeline
print('Ø Annahmen pro Narrativ nach Pipeline:')
display(
    assumption_df.groupby('pipeline_label').size().div(10).rename('Ø Annahmen').round(1)
)

print()
print('Beispiel-Annahmen (erste 9 Einträge):')
print('=' * 70)
for _, r in assumption_df.head(9).iterrows():
    print(f"[{r['pipeline_label']:10s} | {r['xai_model']} | inst={r['instance_id']:4d} | {r['feature']}]")
    print(f"  {r['assumption']}")
    print()

Extrahierte Annahmen gesamt: 273
Ø Annahmen pro Narrativ:     4.55
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval08_ichmoukhamedov/assumptions.csv

Ø Annahmen pro Narrativ nach Pipeline:


pipeline_label
JSON→Text     9.8
Tool-Use      6.8
Vision       10.7
Name: Ø Annahmen, dtype: float64


Beispiel-Annahmen (erste 9 Einträge):
[JSON→Text  | XGB | inst= 224 | hr]
  Der Abend nach der Hauptverkehrszeit zieht Pendler und Freizeitnutzer an, was die Nachfrage erhöht.

[JSON→Text  | XGB | inst= 224 | temp]
  Warme Sommerabende laden erfahrungsgemäß viele Menschen zum Radfahren ein.

[JSON→Text  | XGB | inst= 224 | mnth]
  Der Hochsommer (Juli) gehört zur nachfragestärksten Jahreszeit im Fahrradverleih.

[JSON→Text  | XGB | inst= 224 | yr]
  Im Jahr 2011 war das System noch weniger bekannt und genutzt als im Folgejahr 2012.

[JSON→Text  | XGB | inst= 580 | hr]
  Mitternacht (Stunde 0) ist die nachfrageärmste Zeit des Tages, da kaum Menschen nachts spontan Fahrräder ausleihen.

[JSON→Text  | XGB | inst= 580 | mnth]
  Januar ist der kälteste Wintermonat mit der geringsten Fahrradnachfrage im Jahresverlauf.

[JSON→Text  | XGB | inst= 580 | temp]
  Niedrige Temperaturen um 9 °C schrecken potenzielle Fahrradmieter ab, insbesondere in der Nacht.

[JSON→Text  | XGB | inst=1041 | week

### 5.1 · Perplexität der Annahmen (Gl. 2 im Paper — optional)

Das Paper berechnet Perplexität:
$$\text{PPL} = \left(\prod_{i=0}^{N-1} p_i\right)^{-1/N}$$

relativ zu Llama-3-8B oder Mistral-7B (quantisiert).
**Voraussetzungen:**
```bash
pip install transformers accelerate bitsandbytes
```
Das Modell benötigt ~16 GB RAM oder GPU mit genügend VRAM.

In [9]:
# Perplexitäts-Berechnung — setzt lokales LLM voraus (Gl. 2, Ichmoukhamedov et al.)

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

if not HAS_TRANSFORMERS:
    print('⚠️  transformers nicht installiert — Perplexitäts-Berechnung übersprungen.')
    print('   Zum Aktivieren: pip install transformers accelerate bitsandbytes')
else:
    import torch
    device = ('mps' if torch.backends.mps.is_available()
              else 'cuda' if torch.cuda.is_available() else 'cpu')
    if device == 'cpu':
        print('⚠️  Kein GPU/MPS — Perplexitäts-Berechnung übersprungen (zu langsam auf CPU).')
        HAS_TRANSFORMERS = False

if HAS_TRANSFORMERS:
    PPL_MODEL_ID = 'mistralai/Mistral-7B-v0.3'   # Alternativ: 'meta-llama/Meta-Llama-3-8B'
    print(f'Lade {PPL_MODEL_ID} auf {device} (4-bit quantisiert)...')
    _tok = AutoTokenizer.from_pretrained(PPL_MODEL_ID)
    _mdl = AutoModelForCausalLM.from_pretrained(
        PPL_MODEL_ID, device_map=device, load_in_4bit=True
    )

    def compute_perplexity(text: str) -> float:
        """Perplexität nach Gl. 2 aus dem Paper."""
        inputs = _tok(text, return_tensors='pt').to(device)
        with torch.no_grad():
            loss = _mdl(**inputs, labels=inputs['input_ids']).loss
        return float(torch.exp(loss).item())

    ppl_rows = []
    for _, r in assumption_df.iterrows():
        ppl = compute_perplexity(r['assumption'])
        ppl_rows.append({'pipeline_label': r['pipeline_label'], 'feature': r['feature'], 'ppl': ppl})

    ppl_df = pd.DataFrame(ppl_rows)
    print('\nØ Perplexität der Annahmen nach Pipeline (niedriger = plausibler):')
    display(ppl_df.groupby('pipeline_label')['ppl'].agg(['mean', 'std']).round(1))

    ppl_df.to_csv(OUT_DIR / 'assumption_perplexity.csv', index=False)

⚠️  transformers nicht installiert — Perplexitäts-Berechnung übersprungen.
   Zum Aktivieren: pip install transformers accelerate bitsandbytes


## 6 · Human Similarity

Das Paper vergleicht LLM-generierte Narrative mit **menschlich geschriebenen Referenznarrativen**
mittels Kosinus-Ähnlichkeit auf State-of-the-Art Text-Embeddings (voyage-large-2-instruct).

**Limitation dieser Arbeit:** Keine menschlich geschriebenen Referenznarrative verfügbar.
Daher: Human-Similarity-Metrik nicht berechenbar.

**Proxy-Metrik:** Cross-Pipeline-Ähnlichkeit — misst Konsistenz der drei Pipelines
für dieselbe Instanz (kein direktes Äquivalent im Paper, aber informativ).

In [10]:
try:
    from sentence_transformers import SentenceTransformer
    import torch
    HAS_ST = True
except ImportError:
    HAS_ST = False

if not HAS_ST:
    print('⚠️  sentence-transformers nicht installiert.')
    print('   Human Similarity und Cross-Pipeline-Ähnlichkeit übersprungen.')
    print('   Zum Aktivieren: pip install sentence-transformers')
else:
    print('Lade Embedding-Modell (all-MiniLM-L6-v2)...')
    emb_model = SentenceTransformer('all-MiniLM-L6-v2')

    from sklearn.metrics.pairwise import cosine_similarity

    cross_rows = []
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            group = df[(df.xai_model == xai.upper()) & (df.instance_id == iid)]
            texts = [row['explanation'] for _, row in group.iterrows()]
            if len(texts) < 2:
                continue
            embs = emb_model.encode(texts)
            sims = cosine_similarity(embs)
            # Ø off-diagonal = paarweise Ähnlichkeit der drei Pipelines
            n = len(texts)
            off_diag = [(sims[i, j]) for i in range(n) for j in range(i+1, n)]
            cross_rows.append({
                'xai_model':  xai.upper(),
                'instance_id': iid,
                'mean_cosine': round(float(np.mean(off_diag)), 4),
            })

    cross_df = pd.DataFrame(cross_rows)
    print('Cross-Pipeline-Kosinus-Ähnlichkeit (1.0 = identisch):')
    display(cross_df.groupby('xai_model')[['mean_cosine']].mean().round(3))
    cross_df.to_csv(OUT_DIR / 'cross_pipeline_similarity.csv', index=False)

⚠️  sentence-transformers nicht installiert.
   Human Similarity und Cross-Pipeline-Ähnlichkeit übersprungen.
   Zum Aktivieren: pip install sentence-transformers


## 7 · Gesamtübersicht

In [11]:
print('=== Evaluation nach Ichmoukhamedov et al. (2024) — Zusammenfassung ===')
print()

faith_summary = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3)
faith_std     = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].std().round(3)

print('Faithfulness (Ø ± Std):')
for pl in [PIPELINE_LABELS[p] for p in PIPELINES]:
    if pl not in faith_summary.index:
        continue
    m = faith_summary.loc[pl]
    s = faith_std.loc[pl]
    def _fmt(col):
        mv = m[col]
        sv = s[col]
        if pd.isna(mv):
            return f'{col}=n/a'
        std_str = f'±{sv:.3f}' if not pd.isna(sv) else ''
        return f'{col}={mv:.3f}{std_str}'
    print(f"  {pl:10s}  {_fmt('RA')}  {_fmt('SA')}  {_fmt('VA')}")

print()
print(f'Assumptions extrahiert: {len(assumption_df)} '
      f'(Ø {len(assumption_df)/len(df):.1f} pro Narrativ)')

print()
print('Implementierungsstatus:')
print('  ✓ Faithfulness (RA, SA, VA)    — implementiert nach Gl. 1')
print('  ✓ Assumptions-Extraktion        — per Extraction LLM (Claude)')
print('  ⚠ Assumptions-Perplexität (PPL) — benötigt lokales LLM (Zelle 15)')
print('  ✗ Human Similarity              — keine Referenznarrative verfügbar')

# Ergebnisse speichern
faith_df.to_csv(OUT_DIR / 'faithfulness_metrics.csv', index=False)
faith_summary.to_csv(OUT_DIR / 'faithfulness_summary.csv')
print(f'\nDateien gespeichert in: {OUT_DIR}')

=== Evaluation nach Ichmoukhamedov et al. (2024) — Zusammenfassung ===

Faithfulness (Ø ± Std):
  JSON→Text   RA=0.846±0.267  SA=0.988±0.056  VA=1.000±0.000
  Vision      RA=0.788±0.332  SA=0.983±0.075  VA=1.000±0.000
  Tool-Use    RA=0.883±0.217  SA=1.000±0.000  VA=1.000±0.000

Assumptions extrahiert: 273 (Ø 4.5 pro Narrativ)

Implementierungsstatus:
  ✓ Faithfulness (RA, SA, VA)    — implementiert nach Gl. 1
  ✓ Assumptions-Extraktion        — per Extraction LLM (Claude)
  ⚠ Assumptions-Perplexität (PPL) — benötigt lokales LLM (Zelle 15)
  ✗ Human Similarity              — keine Referenznarrative verfügbar

Dateien gespeichert in: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval08_ichmoukhamedov
